# Fake News Detection & Retrieval: Milestone 3 - Ranked Retrieval Evaluation
**Course:** SPPU Information Retrieval (IR) Mini Project
**Algorithms:** TF-IDF Cosine Dot Product vs BM25 Okapi Two-Stage Retrieval


## 1. Setup & Load Serving Matrix


In [ ]:
import sys
import gzip
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

from ml.config import MODELS, REPORTS
from ml.engine import tfidf_topk, bm25_topk, calculate_neighbour_vote

# Load models and serving matrix
import joblib
vec = joblib.load(MODELS / "tfidf_vectorizer.joblib")
serving_matrix = sparse.load_npz(MODELS / "serving_matrix.npz").tocsr()

with gzip.open(MODELS / "serving_meta.json.gz", "rt", encoding="utf-8") as f:
    serving_meta = json.load(f)

corpus_tokens = [doc["clean"].split() for doc in serving_meta]
labels = np.array([doc["label"] for doc in serving_meta])
print(f"Loaded serving corpus: {serving_matrix.shape[0]} documents, {serving_matrix.shape[1]} terms.")


## 2. Querying the Ranked Retrieval Engine


In [ ]:
query = "Presidential election results certified following bipartisan vote in senate"
from ml.preprocess import preprocess
q_clean = preprocess(query)
q_vec = vec.transform([q_clean])

print("Query:", query)
print("Cleaned Query:", q_clean)

# Rank with TF-IDF Cosine
top_idx, scores = tfidf_topk(q_vec, serving_matrix, k=5)
print("\n--- TF-IDF Cosine Top-5 ---")
for r, (idx, score) in enumerate(zip(top_idx, scores), 1):
    doc = serving_meta[idx]
    lbl = "Real" if doc["label"] == 1 else "Fake"
    print(f"#{r} [Score: {score:.3f} | {lbl}] {doc['title']}")


## 3. Comparing with Probabilistic BM25 Okapi


In [ ]:
bm_idx, bm_scores = bm25_topk(q_clean.split(), corpus_tokens, serving_matrix, q_vec, k=5)
print("--- BM25 Okapi Top-5 ---")
for r, (idx, score) in enumerate(zip(bm_idx, bm_scores), 1):
    doc = serving_meta[idx]
    lbl = "Real" if doc["label"] == 1 else "Fake"
    print(f"#{r} [BM25: {score:.2f} | {lbl}] {doc['title']}")


## 4. Quantitative IR Evaluation: Label-Consistency@K and MRR
We evaluate on 200 random test queries against the indexed corpus to measure retrieval precision proxy.


In [ ]:
# Sample 200 queries
np.random.seed(42)
test_indices = np.random.choice(len(serving_meta), size=200, replace=False)

tfidf_p5 = []
bm25_p5 = []

for q_idx in test_indices:
    q_doc = serving_meta[q_idx]
    true_lbl = q_doc["label"]
    q_tokens = corpus_tokens[q_idx]
    q_vec = serving_matrix[q_idx]
    
    # TF-IDF Top-5 (excluding self at rank 0)
    idx_t, _ = tfidf_topk(q_vec, serving_matrix, k=6)
    idx_t = idx_t[idx_t != q_idx][:5]
    if len(idx_t) > 0:
        tfidf_p5.append(np.mean(labels[idx_t] == true_lbl))
        
    # BM25 Top-5
    idx_b, _ = bm25_topk(q_tokens, corpus_tokens, serving_matrix, q_vec, k=6)
    idx_b = idx_b[idx_b != q_idx][:5]
    if len(idx_b) > 0:
        bm25_p5.append(np.mean(labels[idx_b] == true_lbl))

print(f"TF-IDF Label-Consistency@5: {np.mean(tfidf_p5):.4f}")
print(f"BM25   Label-Consistency@5: {np.mean(bm25_p5):.4f}")
